In [1]:
import sys
sys.path.append("../")

In [2]:
from src.llm.prompt import system_prompt
from pathlib import Path
from src.llm.loading import Loader
from src.llm.models import Model

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = Model(name="Qwen/Qwen2.5-1.5B-Instruct")
model

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 11029.14it/s]


Model(name='Qwen/Qwen2.5-1.5B-Instruct')

In [4]:
loader = Loader(path=Path("../data/data.json"), test_k=0.2, seed=42)
tests = loader.load_test_dataset()
tests

Dataset({
    features: ['input', 'target'],
    num_rows: 10
})

In [5]:
system_prompt.render()

'You mask personal information in text.\nKeep all non-sensitive text unchanged.\nDo not paraphrase.\nReturn only the masked text.'

In [6]:
prompt_messages = [
            {"role": "system", "content": system_prompt.render()},
            {"role": "user", "content": tests[0]["input"]},
        ]
print(prompt_messages)

[{'role': 'system', 'content': 'You mask personal information in text.\nKeep all non-sensitive text unchanged.\nDo not paraphrase.\nReturn only the masked text.'}, {'role': 'user', 'content': "Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com."}]


In [7]:
tokenizer = model.tokenizer
prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
print(prompt_text)   

<|im_start|>system
You mask personal information in text.
Keep all non-sensitive text unchanged.
Do not paraphrase.
Return only the masked text.<|im_end|>
<|im_start|>user
Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com.<|im_end|>
<|im_start|>assistant



In [8]:
target_text = tests[0]["target"] + tokenizer.eos_token
target_text

"Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].<|im_end|>"

In [9]:
tokenizer(prompt_text, add_special_tokens=False)

{'input_ids': [151644, 8948, 198, 2610, 6911, 4345, 1995, 304, 1467, 624, 19434, 678, 2477, 56667, 1467, 34857, 624, 5404, 537, 62230, 10632, 624, 5598, 1172, 279, 42148, 1467, 13, 151645, 198, 151644, 872, 198, 9707, 17618, 13, 22278, 11, 646, 498, 1779, 3170, 847, 3213, 8317, 315, 400, 17, 20, 15, 12492, 944, 1012, 15233, 30, 3017, 2692, 1372, 374, 220, 20, 21, 22, 23, 24, 15, 323, 847, 2551, 374, 18740, 276, 1418, 36808, 35487, 905, 13, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [10]:
prompt_token_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
target_token_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]

In [11]:
MAX_VAL = 256
diff = MAX_VAL - len(target_token_ids)

In [12]:
input_token_ids = prompt_token_ids[:diff] + target_token_ids
len(input_token_ids)

119

In [13]:
labels = [-100] * len(prompt_token_ids[:diff]) + target_token_ids
labels[:10]

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

In [14]:
attantion_mask = [1] * len(input_token_ids)
attantion_mask[:10]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [15]:
(len(input_token_ids), len(labels), len(attantion_mask))

(119, 119, 119)

In [16]:
pad_len = MAX_VAL - len(input_token_ids)
pad_len

137

In [17]:
tokenizer.pad_token_id

151643

In [18]:
input_token_ids = input_token_ids + (pad_len * [tokenizer.pad_token_id])
input_token_ids[:10]

[151644, 8948, 198, 2610, 6911, 4345, 1995, 304, 1467, 624]

In [19]:
labels = labels + ([-100] * pad_len)
labels[:10]

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

In [20]:
attantion_mask = attantion_mask + ([0] * pad_len)
attantion_mask[:10]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [21]:
from src.llm.processing import Processor

preprocessor = Processor(
    tokenizer=tokenizer,
    max_val=256
)

preprocessor.preprocess(tests[1])

{'input_ids': [151644,
  8948,
  198,
  2610,
  6911,
  4345,
  1995,
  304,
  1467,
  624,
  19434,
  678,
  2477,
  56667,
  1467,
  34857,
  624,
  5404,
  537,
  62230,
  10632,
  624,
  5598,
  1172,
  279,
  42148,
  1467,
  13,
  151645,
  198,
  151644,
  872,
  198,
  30665,
  16064,
  13,
  37710,
  11,
  358,
  1184,
  311,
  1895,
  264,
  5558,
  45353,
  3701,
  13,
  3017,
  4540,
  1372,
  374,
  320,
  20,
  20,
  20,
  8,
  220,
  22,
  23,
  24,
  12,
  15,
  16,
  17,
  18,
  323,
  847,
  2692,
  1372,
  374,
  220,
  21,
  22,
  23,
  24,
  15,
  16,
  13,
  151645,
  198,
  151644,
  77091,
  198,
  30665,
  508,
  56588,
  1125,
  358,
  1184,
  311,
  1895,
  264,
  5558,
  45353,
  3701,
  13,
  3017,
  4540,
  1372,
  374,
  508,
  74998,
  19364,
  60,
  323,
  847,
  2692,
  1372,
  374,
  508,
  73383,
  19364,
  936,
  151645,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
